# Download Large Files

Downloads `files.zip` from Google Drive, extracts it to the project root, then deletes the zip.

**Contents of files.zip:**
- `Leo/Other Outputs/arxiv_embeddings.npz`
- `Xiong's output/data/arxiv-metadata-oai-snapshot.json`
- `Xiong's output/data/citation_results.jsonl`
- `Xiong's output/outputs/arxiv_ids_all.csv`
- `Xiong's output/outputs/arxiv_metadata_clean.parquet`
- `Xiong's output/outputs/arxiv_with_citations.parquet`

In [5]:
%pip install -q gdown tqdm

Note: you may need to restart the kernel to use updated packages.


In [6]:
import platform, os
from pathlib import Path

OS = platform.system()  # 'Windows' or 'Darwin' (macOS)
ROOT = Path(os.path.dirname(os.path.abspath("__file__")))
ZIP_PATH = ROOT / "files.zip"
FILE_ID = "1QPbh5S4w9RDdgy4YRCZO0kKMp9fBgv2f"

print(f"OS      : {OS}")
print(f"Project : {ROOT}")
print(f"Zip out : {ZIP_PATH}")

OS      : Windows
Project : c:\github_repo\MACS37005_Final_Project
Zip out : c:\github_repo\MACS37005_Final_Project\files.zip


## Step 1 — Download files.zip

In [7]:
import gdown

if ZIP_PATH.exists():
    print(f"[SKIP] files.zip already exists ({ZIP_PATH.stat().st_size / 1024**2:.0f} MB). Delete it to re-download.")
else:
    print("Starting download...")
    gdown.download(id=FILE_ID, output=str(ZIP_PATH), fuzzy=True)
    print(f"\nDownloaded: {ZIP_PATH.stat().st_size / 1024**2:.1f} MB")

Starting download...


Downloading...
From (original): https://drive.google.com/uc?id=1QPbh5S4w9RDdgy4YRCZO0kKMp9fBgv2f
From (redirected): https://drive.google.com/uc?id=1QPbh5S4w9RDdgy4YRCZO0kKMp9fBgv2f&confirm=t&uuid=e8e89816-8d28-4e0c-9236-6550ba515b16
To: c:\github_repo\MACS37005_Final_Project\files.zip
100%|██████████| 6.35G/6.35G [02:36<00:00, 40.7MB/s]


Downloaded: 6059.7 MB


## Step 2 — Extract and clean up

In [8]:
import zipfile
from tqdm.notebook import tqdm

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"{ZIP_PATH.name} not found — run Step 1 first.")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    members = zf.infolist()

    # macOS: strip __MACOSX metadata entries
    if OS == "Darwin":
        members = [m for m in members if not m.filename.startswith("__MACOSX")]

    total_bytes = sum(m.file_size for m in members)
    print(f"Extracting {len(members)} entries ({total_bytes / 1024**2:.0f} MB uncompressed) to:")
    print(f"  {ROOT}")

    with tqdm(total=total_bytes, unit="B", unit_scale=True,
              unit_divisor=1024, dynamic_ncols=True, desc="Extracting") as bar:
        for member in members:
            zf.extract(member, ROOT)
            bar.update(member.file_size)

print("Extraction complete.")

ZIP_PATH.unlink()
print(f"Deleted files.zip")

Extracting 506 entries (9586 MB uncompressed) to:
  c:\github_repo\MACS37005_Final_Project


Extracting:   0%|          | 0.00/9.36G [00:00<?, ?B/s]

Extraction complete.
Deleted files.zip


## Step 3 — Verify

In [9]:
EXPECTED = [
    "Leo/Other Outputs/arxiv_embeddings.npz",
    "Xiong's output/data/arxiv-metadata-oai-snapshot.json",
    "Xiong's output/data/citation_results.jsonl",
    "Xiong's output/outputs/arxiv_ids_all.csv",
    "Xiong's output/outputs/arxiv_metadata_clean.parquet",
    "Xiong's output/outputs/arxiv_with_citations.parquet",
]

print("Verification:")
all_ok = True
for rel in EXPECTED:
    p = ROOT / rel
    if p.exists():
        print(f"  OK    {rel}  ({p.stat().st_size / 1024**2:.1f} MB)")
    else:
        print(f"  FAIL  {rel}  — NOT FOUND")
        all_ok = False

print("\nAll files present!" if all_ok else "\nSome files are missing — check the zip structure.")

Verification:
  OK    Leo/Other Outputs/arxiv_embeddings.npz  (4037.1 MB)
  OK    Xiong's output/data/arxiv-metadata-oai-snapshot.json  (4892.2 MB)
  OK    Xiong's output/data/citation_results.jsonl  (137.7 MB)
  OK    Xiong's output/outputs/arxiv_ids_all.csv  (34.3 MB)
  OK    Xiong's output/outputs/arxiv_metadata_clean.parquet  (32.1 MB)
  OK    Xiong's output/outputs/arxiv_with_citations.parquet  (35.7 MB)

All files present!
